In [42]:
import pandas as pd
import numpy as np

In [43]:
df = pd.read_csv("behavior_preprocessed_data.csv")

In [44]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,is_weekend,usual_country,usual_device,usual_browser,max_normal_failed_attempts,usual_start_hour,usual_end_hour
0,user_1,2026-04-26 13:43:59,UK,iPhone,Safari,1,Success,0,13,6,1,UK,iPhone,Safari,16,0,23
1,user_1,2026-04-26 20:15:15,UK,iPhone,Safari,3,Success,0,20,6,1,UK,iPhone,Safari,16,0,23
2,user_1,2026-04-27 15:38:08,UK,iPhone,Safari,1,Success,0,15,0,0,UK,iPhone,Safari,16,0,23
3,user_1,2026-04-27 21:01:05,UK,iPhone,Safari,1,Success,0,21,0,0,UK,iPhone,Safari,16,0,23
4,user_1,2026-04-27 21:37:38,UK,iPhone,Safari,0,Success,0,21,0,0,UK,iPhone,Safari,16,0,23


In [45]:
df["is_new_country"] = (
    df["country"] != df["usual_country"]
).astype(int)

In [46]:
df["is_new_device"] = (
    df["device_type"] != df["usual_device"]
).astype(int)

In [47]:
df["is_new_browser"] = (
    df["browser"] != df["usual_browser"]
).astype(int)

In [48]:
df["unusual_login_hour"] = (

    (df["hour"] < df["usual_start_hour"]) |

    (df["hour"] > df["usual_end_hour"])

).astype(int)

In [49]:
df["excessive_failed_attempts"] = (

    df["failed_attempts"] >

    (df["max_normal_failed_attempts"] + 2)

).astype(int)

In [50]:
df["behavior_risk_score"] = (

    df["is_new_country"] +

    df["is_new_device"] +

    df["is_new_browser"] +

    df["unusual_login_hour"] +

    df["excessive_failed_attempts"]

)

In [51]:
df["high_risk_login"] = (
    df["behavior_risk_score"] >= 3
).astype(int)

In [52]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,...,max_normal_failed_attempts,usual_start_hour,usual_end_hour,is_new_country,is_new_device,is_new_browser,unusual_login_hour,excessive_failed_attempts,behavior_risk_score,high_risk_login
0,user_1,2026-04-26 13:43:59,UK,iPhone,Safari,1,Success,0,13,6,...,16,0,23,0,0,0,0,0,0,0
1,user_1,2026-04-26 20:15:15,UK,iPhone,Safari,3,Success,0,20,6,...,16,0,23,0,0,0,0,0,0,0
2,user_1,2026-04-27 15:38:08,UK,iPhone,Safari,1,Success,0,15,0,...,16,0,23,0,0,0,0,0,0,0
3,user_1,2026-04-27 21:01:05,UK,iPhone,Safari,1,Success,0,21,0,...,16,0,23,0,0,0,0,0,0,0
4,user_1,2026-04-27 21:37:38,UK,iPhone,Safari,0,Success,0,21,0,...,16,0,23,0,0,0,0,0,0,0


In [53]:
df.corr(numeric_only=True)["is_anomaly"].sort_values(
    ascending=False
)

is_anomaly                    1.000000
is_new_device                 1.000000
is_new_browser                1.000000
behavior_risk_score           1.000000
is_new_country                1.000000
high_risk_login               1.000000
failed_attempts               0.875090
max_normal_failed_attempts    0.049084
usual_end_hour               -0.003277
is_weekend                   -0.007223
day_of_week                  -0.010835
usual_start_hour             -0.024446
hour                         -0.063254
unusual_login_hour                 NaN
excessive_failed_attempts          NaN
Name: is_anomaly, dtype: float64

In [54]:
columns_to_drop = [

    "timestamp",

    "usual_country",

    "usual_device",

    "usual_browser"

]

df = df.drop(columns=columns_to_drop)

In [55]:
df.head()

,user_id,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,is_weekend,max_normal_failed_attempts,usual_start_hour,usual_end_hour,is_new_country,is_new_device,is_new_browser,unusual_login_hour,excessive_failed_attempts,behavior_risk_score,high_risk_login
0,user_1,UK,iPhone,Safari,1,Success,0,13,6,1,16,0,23,0,0,0,0,0,0,0
1,user_1,UK,iPhone,Safari,3,Success,0,20,6,1,16,0,23,0,0,0,0,0,0,0
2,user_1,UK,iPhone,Safari,1,Success,0,15,0,0,16,0,23,0,0,0,0,0,0,0
3,user_1,UK,iPhone,Safari,1,Success,0,21,0,0,16,0,23,0,0,0,0,0,0,0
4,user_1,UK,iPhone,Safari,0,Success,0,21,0,0,16,0,23,0,0,0,0,0,0,0


In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   user_id                     10000 non-null  str  
 1   country                     10000 non-null  str  
 2   device_type                 10000 non-null  str  
 3   browser                     10000 non-null  str  
 4   failed_attempts             10000 non-null  int64
 5   login_status                10000 non-null  str  
 6   is_anomaly                  10000 non-null  int64
 7   hour                        10000 non-null  int64
 8   day_of_week                 10000 non-null  int64
 9   is_weekend                  10000 non-null  int64
 10  max_normal_failed_attempts  10000 non-null  int64
 11  usual_start_hour            10000 non-null  int64
 12  usual_end_hour              10000 non-null  int64
 13  is_new_country              10000 non-null  int64
 14  is_new_device     

In [59]:
# -----------------------------------
# LABEL ENCODING
# -----------------------------------

from sklearn.preprocessing import LabelEncoder

import joblib

country_encoder = LabelEncoder()

device_encoder = LabelEncoder()

browser_encoder = LabelEncoder()

# -----------------------------------
# ENCODE COLUMNS
# -----------------------------------

df["country"] = country_encoder.fit_transform(
    df["country"]
)

df["device_type"] = device_encoder.fit_transform(
    df["device_type"]
)

df["browser"] = browser_encoder.fit_transform(
    df["browser"]
)
user_encoder = LabelEncoder()

df["user_id"] = user_encoder.fit_transform(
    df["user_id"]
)
status_encoder = LabelEncoder()

df["login_status"] = status_encoder.fit_transform(
    df["login_status"]
)

joblib.dump(
    status_encoder,
    "status_encoder.pkl"
)

# -----------------------------------
# SAVE ENCODERS
# -----------------------------------

joblib.dump(

    country_encoder,

    "country_encoder.pkl"
)

joblib.dump(

    device_encoder,

    "device_encoder.pkl"
)

joblib.dump(

    browser_encoder,

    "browser_encoder.pkl"
)
joblib.dump(

    user_encoder,

    "user_encoder.pkl"
)
print(df.head())

   user_id  country  device_type  browser  failed_attempts  login_status  \
0        0        4            4        3                1             1   
1        0        4            4        3                3             1   
2        0        4            4        3                1             1   
3        0        4            4        3                1             1   
4        0        4            4        3                0             1   

   is_anomaly  hour  day_of_week  is_weekend  max_normal_failed_attempts  \
0           0    13            6           1                          16   
1           0    20            6           1                          16   
2           0    15            0           0                          16   
3           0    21            0           0                          16   
4           0    21            0           0                          16   

   usual_start_hour  usual_end_hour  is_new_country  is_new_device  \
0               

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   user_id                     10000 non-null  int64
 1   country                     10000 non-null  int64
 2   device_type                 10000 non-null  int64
 3   browser                     10000 non-null  int64
 4   failed_attempts             10000 non-null  int64
 5   login_status                10000 non-null  int64
 6   is_anomaly                  10000 non-null  int64
 7   hour                        10000 non-null  int64
 8   day_of_week                 10000 non-null  int64
 9   is_weekend                  10000 non-null  int64
 10  max_normal_failed_attempts  10000 non-null  int64
 11  usual_start_hour            10000 non-null  int64
 12  usual_end_hour              10000 non-null  int64
 13  is_new_country              10000 non-null  int64
 14  is_new_device     

In [61]:
df.head()

,user_id,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,is_weekend,max_normal_failed_attempts,usual_start_hour,usual_end_hour,is_new_country,is_new_device,is_new_browser,unusual_login_hour,excessive_failed_attempts,behavior_risk_score,high_risk_login
0,0,4,4,3,1,1,0,13,6,1,16,0,23,0,0,0,0,0,0,0
1,0,4,4,3,3,1,0,20,6,1,16,0,23,0,0,0,0,0,0,0
2,0,4,4,3,1,1,0,15,0,0,16,0,23,0,0,0,0,0,0,0
3,0,4,4,3,1,1,0,21,0,0,16,0,23,0,0,0,0,0,0,0
4,0,4,4,3,0,1,0,21,0,0,16,0,23,0,0,0,0,0,0,0


In [62]:
df.to_csv(
    "behavior_feature_engineered_data.csv",
    index=False
)

print("Behavioral Feature Engineering Complete!")

Behavioral Feature Engineering Complete!
